# Lekce 10 - AI agenti v produkci

V této lekci se naučíte **produkční vzory** pro AI agenty pomocí Microsoft Agent Framework (Python). Pokryjeme:

- **Sledovatelnost** — přidání časování a protokolování interakcí agenta
- **Hodnocení** — použití hodnotícího agenta k ohodnocení kvality odpovědí
- **Řízení nákladů** — strategie pro optimalizaci tokenů a výběr modelu

Scénář je **cestovní agent**, který pomáhá uživatelům plánovat cesty, s monitorováním a hodnocením navrstveným navrch.


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Provozní úvahy

Přechod AI agentů z prototypů do produkce vyžaduje pečlivou pozornost třem pilířům:

1. **Sledovatelnost** — Potřebujete přehled o tom, co agent dělá, jak dlouho to trvá a jaké nástroje volá. Bez trasování a logování je ladění produkčních problémů téměř nemožné.

2. **Hodnocení** — Automatizované kontroly kvality zajišťují, že odpovědi agenta zůstávají přesné, úplné a užitečné v čase. Hodnotící agent může ohodnotit odpovědi podle definovaných kritérií.

3. **Řízení nákladů** — Spotřeba tokenů přímo ovlivňuje náklady. Strategie jako optimalizace promptu, výběr modelu a caching pomáhají udržet výdaje pod kontrolou bez obětování kvality.


## Vytváření pozorovatelného agenta

Definujeme nástroje pro cestování a obalíme volání agenta časováním, abychom mohli sledovat latenci. V produkci byste integrovali OpenTelemetry nebo podobný systém trasování.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie řízení nákladů

Kontrola nákladů je pro produkční AI agenty kritická. Zde jsou klíčové strategie:

| Strategie | Popis |
|---|---|
| **Optimalizace promptů** | Udržujte systémové instrukce stručné. Odstraňte nadbytečný kontext k redukci vstupních tokenů. |
| **Výběr modelu** | Používejte menší, levnější modely (např. GPT-5-mini) pro jednoduché úkoly jako klasifikace nebo extrakce a větší modely rezervujte pro složité uvažování. |
| **Caching** | Ukládejte do mezipaměti výsledky nástrojů a časté dotazy, abyste se vyhnuli opakovaným voláním API. |
| **Tokenové limity** | Nastavte limity `max_tokens`, aby se zabránilo neočekávaně dlouhým odpovědím. |
| **Seskupování požadavků** | Seskupujte více uživatelských dotazů do jednoho API volání, pokud je to možné. |

V praxi dobře funguje vrstvený přístup: směrujte jednoduché požadavky k rychlému, levnému modelu a složité dotazy eskalujte na výkonnější model.


## Shrnutí

V této lekci jste se naučili, jak:

1. **Přidat pozorovatelnost** do interakcí agentů pomocí časování a protokolování, což vytváří základy pro sledování a monitorování.
2. **Automaticky hodnotit odpovědi agentů** pomocí hodnotícího agenta, který skóruje úplnost, přesnost a užitečnost.
3. **Řídit náklady** pomocí optimalizace výzev, výběru modelu, mezipaměti a rozpočtů tokenů.

Tyto vzory pro produkční nasazení pomáhají zajistit, aby vaši AI agenti byli spolehliví, měřitelní a nákladově efektivní ve velkém měřítku.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
